# UQRoute-TC — saved alignment re-audit

The 24 saved Qwen 1.5B development responses are rechecked with the expanded APIBank alignment rule. The audit records token evidence, parser agreement, eligibility changes, and score stability against the previous audit.


## 1. Source revisions and saved run


In [ ]:
from google.colab import drive, files
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

drive.mount('/content/drive')
output_dir = Path('/content/drive/MyDrive/UQRoute-TC/pilots/qwen25-15b-xml-json-dev-24-v1')
for filename in ('manifest.json', 'alignment_coverage_audit_v1.json'):
    if not (output_dir / filename).is_file():
        raise FileNotFoundError(output_dir / filename)

benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
code_revision = 'd1be571d7cb65802ecc77d7440bc9dc4a98851c0'
benchmark = Path('/content/robustbench-tc-alignment-reaudit-v2')
project = Path('/content/uqroute-tc-alignment-reaudit-v2')
for path, url, revision in (
    (benchmark, 'https://github.com/WillChow66/robustbench-tc-release.git', benchmark_revision),
    (project, 'https://github.com/RavindraSSK/uqroute-tc.git', code_revision),
):
    if not path.exists():
        subprocess.run(['git', 'clone', '--no-checkout', url, str(path)], check=True)
    elif subprocess.check_output(
        ['git', '-C', str(path), 'status', '--porcelain'], text=True
    ).strip():
        raise RuntimeError(f'Local changes in {path}')
    subprocess.run(['git', '-C', str(path), 'fetch', 'origin', revision], check=True)
    subprocess.run(['git', '-C', str(path), 'checkout', '--detach', revision], check=True)
    current = subprocess.check_output(
        ['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True
    ).strip()
    assert current == revision

missing = [name for name in ('openai', 'tqdm') if importlib.util.find_spec(name) is None]
if missing:
    requirements = {'openai': 'openai>=1.50,<3', 'tqdm': 'tqdm>=4.66,<5'}
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q']
        + [requirements[name] for name in missing], check=True
    )
print('Alignment revision:', code_revision)
print('Saved run:', output_dir)


## 2. Token and call alignment


In [ ]:
audit_source = r'''
import hashlib
import json
import math
import subprocess
import sys
from collections import Counter
from pathlib import Path

CODE_REVISION = 'd1be571d7cb65802ecc77d7440bc9dc4a98851c0'
BENCHMARK_REVISION = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
PRIOR_SHA256 = '777a0e8e8f63db1765613f74aa509fd01a413f1caea8b70362aa2b270937586e'
MANIFEST_SHA256 = 'd51ba41039d86f84e8e54f1b6c73f3c2c49ac506f14521259714356e99858bf0'


def audit_one(row, prior, reference_parser):
    from uqroute_tc.inference.pilot import token_evidence
    from uqroute_tc.parsing.canonical import canonical_key, parse_prediction
    from uqroute_tc.uncertainty.alignment import meaningful_token_score

    raw = row['prediction']['raw_output']
    parsed = parse_prediction(raw, row['benchmark'], reference_parser)
    calls = row['prediction']['tool_calls']
    response = row['uqroute']['response']
    choice = response['choices'][0] if response and response.get('choices') else None
    evidence = token_evidence(choice) if choice else {'valid': False, 'reason': 'missing response'}
    try:
        matches = canonical_key(list(parsed.calls)) == canonical_key(calls)
    except ValueError:
        matches = False
    entry = {
        'benchmark': row['benchmark'], 'case_id': row['id'],
        'record_status': row['uqroute']['status'], 'parser_status': parsed.status,
        'parsed_call_count': len(parsed.calls), 'parser_matches_saved': matches,
        'token_evidence_valid': evidence['valid'],
        'meaningful_token_status': 'unavailable', 'reason': None,
        'validation_errors': [],
    }
    errors = entry['validation_errors']
    if not matches or parsed.status == 'parse_failure':
        errors.append('saved calls disagree with pinned parser or parsing failed')
    if entry['record_status'] != 'complete' or not choice or choice.get('finish_reason') != 'stop':
        errors.append('saved response is not complete with a stop finish')
    if not evidence['valid']:
        errors.append(evidence.get('reason') or 'invalid chosen-token evidence')
    if choice and (choice.get('message') or {}).get('content') != raw:
        errors.append('visible response differs from saved prediction')
    for field in (
        'benchmark', 'case_id', 'record_status', 'parser_status',
        'parsed_call_count', 'parser_matches_saved', 'token_evidence_valid',
    ):
        if entry[field] != prior[field]:
            errors.append(f'prior audit differs on {field}')
    if errors:
        entry['reason'] = '; '.join(errors)
    elif parsed.status != 'calls':
        entry['reason'] = f'no trusted parsed call: {parsed.status}'
    else:
        try:
            score = meaningful_token_score(
                raw, calls, choice['logprobs']['content'],
                choice['finish_reason'], row['benchmark']
            )
        except ValueError as exc:
            entry['reason'] = str(exc)
        else:
            entry.update(
                meaningful_token_status='valid',
                selected_token_count=len(score.selected_token_indices),
                boundary_crossing_token_count=len(score.boundary_crossing_token_indices),
                mean_token_nll=score.mean_token_nll,
            )
    return entry


def compare_entries(entries, prior_entries):
    prior_by_id = {entry['case_id']: entry for entry in prior_entries}
    newly_eligible, regressions, changed_scores = [], [], []
    for entry in entries:
        old = prior_by_id[entry['case_id']]
        old_valid = old['meaningful_token_status'] == 'valid'
        new_valid = entry['meaningful_token_status'] == 'valid'
        if new_valid and not old_valid:
            newly_eligible.append(entry['case_id'])
        elif old_valid and not new_valid:
            regressions.append(entry['case_id'])
        elif old_valid and new_valid:
            equal = all(entry[k] == old[k] for k in (
                'selected_token_count', 'boundary_crossing_token_count'
            )) and math.isclose(
                entry['mean_token_nll'], old['mean_token_nll'], rel_tol=1e-12, abs_tol=1e-12
            )
            if not equal:
                changed_scores.append(entry['case_id'])
    return {
        'newly_eligible_case_ids': newly_eligible,
        'regressed_case_ids': regressions,
        'previous_valid_scores_changed': changed_scores,
        'validation_error_cases': sum(bool(e['validation_errors']) for e in entries),
    }


def main():
    project, benchmark, output_dir = map(Path, sys.argv[1:])
    for folder, revision in ((project, CODE_REVISION), (benchmark, BENCHMARK_REVISION)):
        assert subprocess.check_output(
            ['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True
        ).strip() == revision
    sys.path.insert(0, str(project / 'src'))
    sys.path.insert(0, str(benchmark / 'scripts'))
    from run_eval import parse_tool_calls
    from uqroute_tc.inference.pilot import _atomic_json

    manifest_path = output_dir / 'manifest.json'
    prior_path = output_dir / 'alignment_coverage_audit_v1.json'
    assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == MANIFEST_SHA256
    assert hashlib.sha256(prior_path.read_bytes()).hexdigest() == PRIOR_SHA256
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    prior = json.loads(prior_path.read_text(encoding='utf-8'))
    split_path = project / 'docs/splits/group_split_v1.json'
    split = json.loads(split_path.read_text(encoding='utf-8'))
    assert manifest['benchmark_revision'] == split['benchmark_revision'] == BENCHMARK_REVISION
    assert hashlib.sha256(split_path.read_bytes()).hexdigest() == manifest['split_sha256']
    assert manifest['model_id'] == 'Qwen/Qwen2.5-1.5B-Instruct'
    assert manifest['model_revision'] == '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
    assert manifest['mode'] == 'single_prompt_tool_calling'
    assert prior['source_manifest_sha256'] == MANIFEST_SHA256
    case_ids = manifest['case_ids']
    prior_by_id = {entry['case_id']: entry for entry in prior['entries']}
    assert len(case_ids) == len(set(case_ids)) == len(prior_by_id) == len(prior['entries']) == 24
    assert set(case_ids) == set(prior_by_id)

    entries = []
    for case_id in case_ids:
        record_path = output_dir / 'records' / (
            hashlib.sha256(case_id.encode('utf-8')).hexdigest() + '.json'
        )
        row = json.loads(record_path.read_text(encoding='utf-8'))
        assert row['id'] == case_id and row['benchmark'] in {'apibank', 'toolalpaca'}
        assert split['assignments'][case_id] == {
            'benchmark': row['benchmark'], 'population': 'primary', 'split': 'development'
        }
        entry = audit_one(row, prior_by_id[case_id], parse_tool_calls)
        entry['source_record_sha256'] = hashlib.sha256(record_path.read_bytes()).hexdigest()
        entries.append(entry)
    assert Counter(e['benchmark'] for e in entries) == {'apibank': 12, 'toolalpaca': 12}
    summary = {}
    for name in ('apibank', 'toolalpaca'):
        subset = [entry for entry in entries if entry['benchmark'] == name]
        summary[name] = {
            'cases': len(subset),
            'parser_status_counts': dict(Counter(e['parser_status'] for e in subset)),
            'meaningful_token_status_counts': dict(Counter(
                e['meaningful_token_status'] for e in subset
            )),
            'unavailable_reasons': dict(Counter(e['reason'] for e in subset if e['reason'])),
        }
    comparison = compare_entries(entries, prior['entries'])
    report = {
        'scope': '24 saved clean development APIBank and ToolAlpaca cases; alignment re-audit',
        'source_manifest_sha256': MANIFEST_SHA256, 'prior_audit_sha256': PRIOR_SHA256,
        'benchmark_revision': BENCHMARK_REVISION, 'split_sha256': manifest['split_sha256'],
        'source_code_revision': manifest['code_revision'], 'audit_code_revision': CODE_REVISION,
        'audit_script_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        'model_id': manifest['model_id'], 'model_revision': manifest['model_revision'],
        'case_count': len(entries), 'summary': summary, 'comparison': comparison,
        'entries': entries,
        'interpretation_limit': 'Development format eligibility only. The expanded rule follows inspection of these cases. Eligibility is not official correctness or population coverage. Unavailable scores are not imputed.',
    }
    audit_path = output_dir / 'alignment_coverage_audit_v2.json'
    _atomic_json(audit_path, report)
    print('Processed:', len(entries))
    print('Parser agreement:', sum(e['parser_matches_saved'] for e in entries))
    print('Valid token evidence:', sum(e['token_evidence_valid'] for e in entries))
    print('Saved:', audit_path)


if __name__ == '__main__':
    main()
'''

script_path = Path('/content/alignment_reaudit_v2.py')
script_path.write_text(audit_source, encoding='utf-8')
subprocess.run(
    [sys.executable, str(script_path), str(project), str(benchmark), str(output_dir)],
    check=True
)


## 3. Audit comparison


In [ ]:
audit_path = output_dir / 'alignment_coverage_audit_v2.json'
report = json.loads(audit_path.read_text(encoding='utf-8'))
for name, summary in report['summary'].items():
    eligible = summary['meaningful_token_status_counts'].get('valid', 0)
    print(name, '| eligible:', eligible, '/', summary['cases'])
    print('  parser:', summary['parser_status_counts'])
    print('  unavailable:', summary['unavailable_reasons'])
comparison = report['comparison']
print('Validation error cases:', comparison['validation_error_cases'])
print('Newly eligible:', comparison['newly_eligible_case_ids'])
print('Regressions:', comparison['regressed_case_ids'])
print('Previous valid scores changed:', comparison['previous_valid_scores_changed'])
print('Saved:', audit_path)
files.download(str(audit_path))
